In [1]:
import os
import json
import numpy as np
import torch
import pandas as pd
import cv2
import trimesh
from pathlib import Path
from PIL import Image
from scipy.spatial.transform import Rotation

In [2]:
PROJECT_ROOT = Path(".").resolve().parent

GD_CONFIG  = Path.home() / "rai_workspace/grounding_dino_weights/GroundingDINO_SwinT_OGC.py"
GD_WEIGHTS = Path.home() / "rai_workspace/grounding_dino_weights/groundingdino_swint_ogc.pth"

MESH_PATH = PROJECT_ROOT / "megapose_objects/lamp/mesh.ply"
CAM_INFO  = PROJECT_ROOT / "outputs/camera_info.json"

DETECTION_PROMPT     = "yellow lamp"
BOX_THRESHOLD        = 0.30
TEXT_THRESHOLD       = 0.25
N_REFINER_ITERATIONS = 5
DEVICE               = "cuda"

# Known target pose from the .g file (world frame): [x, y, z, qw, qx, qy, qz]
TARGET_POSE_RAI = np.array([0.219547, 0.280159, 0.698313, 0.38692, 0.0, 0.0, -0.922113])

In [3]:
def rai_pose_to_matrix(pose: np.ndarray) -> np.ndarray:
    pos = pose[:3]
    qw, qx, qy, qz = pose[3], pose[4], pose[5], pose[6]
    R = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3]  = pos
    return T


def boxes_cxcywh_to_xyxy(boxes_norm: torch.Tensor, width: int, height: int) -> torch.Tensor:
    """Convert GroundingDINO [cx,cy,w,h] normalised -> pixel [x1,y1,x2,y2]."""
    cx, cy, w, h = boxes_norm.unbind(-1)
    x1 = (cx - w / 2) * width
    y1 = (cy - h / 2) * height
    x2 = (cx + w / 2) * width
    y2 = (cy + h / 2) * height
    return torch.stack([x1, y1, x2, y2], dim=-1)


def project_pt(pt_3d, K):
    p = K @ np.array(pt_3d)
    return (int(p[0] / p[2]), int(p[1] / p[2]))

### 1. Load GroundingDINO

In [4]:
from groundingdino.util.inference import load_model, load_image, predict

print("Loading GroundingDINO …")
gd_model = load_model(str(GD_CONFIG), str(GD_WEIGHTS))
gd_model.eval()
print("Done.")

Loading GroundingDINO …


final text_encoder_type: bert-base-uncased


Done.


### 2. Detect Lamp Across All Cameras

In [5]:
with open(CAM_INFO) as f:
    camera_info = json.load(f)

print(f"Running GroundingDINO with prompt: '{DETECTION_PROMPT}'")

best = {"score": -1.0, "cam": None, "box_xyxy": None}

for cam_name, info in camera_info.items():
    rgb_path = info["rgb_path"]
    W, H     = info["width"], info["height"]

    image_source, image_tensor = load_image(rgb_path)

    boxes_norm, logits, phrases = predict(
        model=gd_model,
        image=image_tensor,
        caption=DETECTION_PROMPT,
        box_threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
    )

    if len(logits) == 0:
        print(f"  {cam_name}: no detection")
        continue

    best_idx   = logits.argmax().item()
    best_score = logits[best_idx].item()
    best_box   = boxes_cxcywh_to_xyxy(boxes_norm[best_idx].unsqueeze(0), W, H)[0]

    print(f"  {cam_name}: score={best_score:.3f}  box={best_box.numpy().astype(int).tolist()}")

    if best_score > best["score"]:
        best.update({"score": best_score, "cam": cam_name, "box_xyxy": best_box})

if best["cam"] is None:
    raise RuntimeError(
        "GroundingDINO found no detections. "
        "Lower BOX_THRESHOLD or change the prompt."
    )

print(f"\nBest detection: {best['cam']}  score={best['score']:.3f}")

Running GroundingDINO with prompt: 'yellow lamp'


  cam_dim_0: score=0.539  box=[255, 1280, 331, 1391]
  cam_dim_1: score=0.510  box=[96, 405, 1824, 769]
  cam_dim_2: score=0.532  box=[97, 423, 1823, 767]
  cam_dim_3: score=0.455  box=[96, 400, 1824, 768]
  cam_dim_4: score=0.452  box=[96, 425, 1823, 767]

Best detection: cam_dim_0  score=0.539


### 3. Load MegaPose

In [6]:
# Jupyter kernels don't inherit ~/.bashrc — set HAPPYPOSE_DATA_DIR explicitly.
# Run `echo $HAPPYPOSE_DATA_DIR` in a terminal on the server to get the value.
if "HAPPYPOSE_DATA_DIR" not in os.environ:
    os.environ["HAPPYPOSE_DATA_DIR"] = "/home/salman/rai_workspace/happypose_data"  # <-- update if different

print("HAPPYPOSE_DATA_DIR:", os.environ["HAPPYPOSE_DATA_DIR"])

HAPPYPOSE_DATA_DIR: /home/salman/rai_workspace/happypose_data


In [7]:
from happypose.toolbox.datasets.object_dataset import RigidObjectDataset, RigidObject
from happypose.toolbox.utils.load_model import load_named_model
from happypose.toolbox.inference.types import ObservationTensor
from happypose.toolbox.utils.tensor_collection import PandasTensorCollection

print("Loading MegaPose …")

object_dataset = RigidObjectDataset([
    RigidObject(
        label="lamp",
        mesh_path=MESH_PATH,
        scaling_factor=1.0,
    )
])

# megapose-1.0-RGBD uses the depth refiner — better accuracy than RGB-only.
pose_estimator = load_named_model(
    "megapose-1.0-RGBD",
    object_dataset=object_dataset,
)
# load_named_model moves sub-models to CUDA but not the PoseEstimator wrapper.
# _SO3_grid is a registered buffer on the wrapper — move it manually.
pose_estimator._SO3_grid = pose_estimator._SO3_grid.to(DEVICE)
print("Done.")

MKL_NUM_THREADS: 1
OMP_NUM_THREADS: 1
CUDA_VISIBLE_DEVICES: 0
EGL_VISIBLE_DEVICES: 0
Loading MegaPose …


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Known pipe types:
  glxGraphicsPipe
(1 aux display modules not yet loaded.)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Known pipe types:
  glxGraphicsPipe
(1 aux display modules not yet loaded.)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid usin

Done.


### 4. Prepare Observation & Run MegaPose

In [8]:
cam_name = best["cam"]
info     = camera_info[cam_name]

rgb   = np.array(Image.open(info["rgb_path"]).convert("RGB"))
depth = np.load(info["depth_path"])
K     = np.array(info["K"], dtype=np.float64)

observation = ObservationTensor.from_numpy(
    rgb=rgb,
    depth=depth,
    K=K,
).to(DEVICE)

box_xyxy = best["box_xyxy"].unsqueeze(0).float().to(DEVICE)

detections = PandasTensorCollection(
    infos=pd.DataFrame({
        "label":       ["lamp"],
        "batch_im_id": [0],
        "instance_id": [0],
        "score":       [float(best["score"])],
    }),
    bboxes=box_xyxy,
)

In [9]:
print("Running MegaPose coarse + refiner …")
output, _ = pose_estimator.run_inference_pipeline(
    observation,
    detections=detections,
    run_detector=False,
    n_refiner_iterations=N_REFINER_ITERATIONS,
    n_pose_hypotheses=1,
)

T_cam_obj = output.poses[0].cpu().numpy()
print("Done.")

Running MegaPose coarse + refiner …
Done.


### 5. Convert to World Frame & Evaluate

In [10]:
T_world_cam = np.array(info["T_world_cam"])
T_world_obj = T_world_cam @ T_cam_obj

position  = T_world_obj[:3, 3]
R_mat     = T_world_obj[:3, :3]
quat_xyzw = Rotation.from_matrix(R_mat).as_quat()        # [x, y, z, w]
quat_wxyz = np.array([quat_xyzw[3], *quat_xyzw[:3]])     # RAI convention

print("=" * 60)
print("ESTIMATED POSE (world frame)")
print("=" * 60)
print(f"Position    [x, y, z] : {position}")
print(f"Quaternion  [w,x,y,z] : {quat_wxyz}")
print(f"\n4x4 matrix:\n{T_world_obj}")

# Compare against ground-truth position from .g file
TRUE_OBJ_POSE_RAI = np.array([-0.620966, -0.342511, 0.671156,
                               0.287543, -0.46152, 0.714029, 0.441])
true_pos = TRUE_OBJ_POSE_RAI[:3]
pos_err  = np.linalg.norm(position - true_pos)

print(f"\nTrue object position (from .g): {true_pos}")
print(f"Estimated position:             {np.round(position, 4)}")
print(f"Position error:                 {pos_err*100:.1f} cm")
print(f"\nTarget position (goal pose):    {TARGET_POSE_RAI[:3]}")

ESTIMATED POSE (world frame)
Position    [x, y, z] : [-0.61998516 -0.34027916  0.68286133]
Quaternion  [w,x,y,z] : [ 0.31881487 -0.49191166  0.67919928  0.44166541]

4x4 matrix:
[[-0.31276    -0.94983113 -0.00144306 -0.61998516]
 [-0.38659304  0.12590916  0.91361517 -0.34027916]
 [-0.8675983   0.28630015 -0.4065775   0.68286133]
 [ 0.          0.          0.          1.        ]]

True object position (from .g): [-0.620966 -0.342511  0.671156]
Estimated position:             [-0.62   -0.3403  0.6829]
Position error:                 1.2 cm

Target position (goal pose):    [0.219547 0.280159 0.698313]


### 6. Visualization

In [11]:
print("Generating visualization …")

vis_img = cv2.imread(info["rgb_path"])   # BGR
H_vis, W_vis = vis_img.shape[:2]
K_vis = np.array(info["K"])

# Green detection bounding box
x1, y1, x2, y2 = best["box_xyxy"].cpu().numpy().astype(int)
cv2.rectangle(vis_img, (x1, y1), (x2, y2), (0, 255, 0), 4)
cv2.putText(vis_img, f"GD score={best['score']:.2f}", (x1, max(y1 - 15, 30)),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

# XYZ axes at estimated pose (red=X, green=Y, blue=Z)
T        = T_cam_obj
axis_len = 0.05  # 5 cm
o_px = project_pt(T[:3, 3], K_vis)
x_px = project_pt(T[:3, 3] + axis_len * T[:3, 0], K_vis)
y_px = project_pt(T[:3, 3] + axis_len * T[:3, 1], K_vis)
z_px = project_pt(T[:3, 3] + axis_len * T[:3, 2], K_vis)
cv2.arrowedLine(vis_img, o_px, x_px, (0, 0, 255), 4, tipLength=0.3)   # X red
cv2.arrowedLine(vis_img, o_px, y_px, (0, 255, 0), 4, tipLength=0.3)   # Y green
cv2.arrowedLine(vis_img, o_px, z_px, (255, 0, 0), 4, tipLength=0.3)   # Z blue

# Project mesh vertices as cyan dots
mesh     = trimesh.load(str(MESH_PATH))
verts    = np.array(mesh.vertices)
verts_hom = np.hstack([verts, np.ones((len(verts), 1))])
verts_cam = (T_cam_obj @ verts_hom.T).T[:, :3]
front     = verts_cam[:, 2] > 0
if front.any():
    pts = (K_vis @ verts_cam[front].T).T
    pts = (pts[:, :2] / pts[:, 2:3]).astype(int)
    in_bounds = (pts[:, 0] >= 0) & (pts[:, 0] < W_vis) & \
                (pts[:, 1] >= 0) & (pts[:, 1] < H_vis)
    for pt in pts[in_bounds][::8]:   # every 8th vertex for speed
        cv2.circle(vis_img, tuple(pt), 3, (255, 255, 0), -1)

# Text overlay with estimated position
pos_str = f"pos=[{position[0]:.3f}, {position[1]:.3f}, {position[2]:.3f}]"
cv2.putText(vis_img, pos_str, (20, H_vis - 30),
            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)
cv2.putText(vis_img, pos_str, (20, H_vis - 30),
            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 1)

out_path = PROJECT_ROOT / "outputs" / "pose_visualization.png"
cv2.imwrite(str(out_path), vis_img)
print(f"Saved -> {out_path}")

Generating visualization …
Saved -> /home/salman/rai_workspace/Voxel_manipulation_v2/pose_estimation/outputs/pose_visualization.png
